# Phase 1 (v2) — Synthetic Patient & Form Data Generation

Generates the v2 dataset for the scannable medical-intake pipeline.

**Outputs (to `data_v2/`):**
- `patient_db_v2.json` — **1000 patients**, each a static identity record plus a `visits[]`
  array (so a patient can be looked up by last-visit date). A visit = `{date_of_visit,
  department, doctor_name, chief_complaint}`.
- `validation_forms_v2.csv` — **100 rows** (first 100 patients), columns ordered exactly
  as the fields appear on the form, for easy hand-writing onto printed blanks.

**Hold-out rule:** for the 100 validation patients, the *current* visit captured in the CSV
is **NOT** written into their db `visits[]`. The app registers it once the patient is matched.

Choice lists (names, departments, allergies, complaints, meds, history, genders) are loaded
from `data_v2/vocab/` — the source vocab, not previously-generated data.

In [ ]:
# Cell 1 — imports & config
import json, csv, random
from pathlib import Path
from datetime import date, timedelta, datetime

random.seed(42)

ROOT  = Path.cwd()
if ROOT.name == "notebooks":          # run from notebooks/ or project root either way
    ROOT = ROOT.parent
DATA  = ROOT / "data"
VOCAB = DATA / "vocab"
GEN   = DATA / "generated"

N_PATIENTS   = 1000
N_VALIDATION = 100              # first 100 patients -> handwriting CSV (current visit held out)
REF_DATE     = date(2026, 6, 21)  # "today" for ages and visit dating

In [ ]:
# Cell 2 — load vocab + form field order
def load(n): return json.load(open(VOCAB / f"{n}.json"))

first_names = load("first_names")
last_names  = load("last_names")
departments = load("departments")
blood_types = load("blood_types")
allergies_v = load("allergies")
complaints  = load("chief_complaints")
med_full    = load("medications")                 # [{name, dosages}]
history_opt = [h for h in load("history_options") if h.lower() != "none"]
genders     = load("genders")

# Column order taken straight from the form's field map, so the CSV reads top-to-bottom
# in the same order a person fills the printed form.
field_map   = json.load(open(DATA / "template" / "field_map.json"))["fields"]
FORM_FIELDS = list(field_map.keys())
print("form field order:")
print(FORM_FIELDS)

In [ ]:
# Cell 3 — field generators
STREETS = ["Maple","Oak","Cedar","Pine","Elm","Washington","Lake","Hill","Park","Sunset",
           "Birch","Walnut","Highland","River","Meadow","Spring","Church","Court","Ridge","Forest"]
SUFFIX  = ["St","Ave","Rd","Blvd","Ln","Dr","Way","Ct"]
CITIES  = ["Riverside","Springfield","Fairview","Madison","Georgetown","Clinton","Franklin",
           "Greenville","Bristol","Ashland","Dover","Salem","Auburn","Milton","Newport"]
STATES  = ["CA","TX","NY","FL","OH","PA","IL","GA","NC","MI","WA","AZ","CO","OR","VA"]
DOMAINS = ["gmail.com","outlook.com","yahoo.com","icloud.com","proton.me"]

def rand_dob(min_age=1, max_age=95):
    age = random.randint(min_age, max_age)
    return REF_DATE.replace(year=REF_DATE.year - age) - timedelta(days=random.randint(0, 364))

def fmt(d): return d.strftime("%d/%m/%Y")
def parse(s): return datetime.strptime(s, "%d/%m/%Y").date()

def age_from(dob):
    return REF_DATE.year - dob.year - ((REF_DATE.month, REF_DATE.day) < (dob.month, dob.day))

def rand_ssn():
    return f"{random.randint(100,899):03d}-{random.randint(10,99):02d}-{random.randint(1,9999):04d}"

def rand_phone():
    return f"{random.randint(200,989):03d}-{random.randint(200,989):03d}-{random.randint(0,9999):04d}"

def rand_insurance():                              # 11 chars, comb groups [3,4,4]
    L = "".join(random.choice("ABCDEFGHJKLMNPRTUVWXYZ") for _ in range(3))
    D = "".join(random.choice("0123456789") for _ in range(8))
    return L + D

def rand_address():
    return (f"{random.randint(10,9999)} {random.choice(STREETS)} {random.choice(SUFFIX)}, "
            f"{random.choice(CITIES)}, {random.choice(STATES)} {random.randint(10000,99999)}")

def make_email(first, last):
    base = "".join(c for c in f"{first}.{last}".lower() if c.isalnum() or c == ".")
    tail = random.choice(["", str(random.randint(1,99))])
    return f"{base}{tail}@{random.choice(DOMAINS)}"

def rand_allergies():
    if random.random() < 0.65: return "None known"
    return ", ".join(random.sample(allergies_v, random.randint(1, 2)))

def rand_history():
    if random.random() < 0.55 or not history_opt: return "None"
    return ", ".join(random.sample(history_opt, random.randint(1, min(3, len(history_opt)))))

def rand_meds():
    if random.random() < 0.55: return "None"
    out = []
    for m in random.sample(med_full, random.randint(1, 2)):
        dose = random.choice(m["dosages"]) if m.get("dosages") else ""
        out.append(f'{m["name"]} {dose}'.strip())
    return ", ".join(out)

def make_visit(dov):
    return {"date_of_visit": fmt(dov),
            "department":     random.choice(departments),
            "doctor_name":    "Dr. " + random.choice(last_names),
            "chief_complaint": random.choice(complaints)}

In [ ]:
# Cell 4 — generate 1000 patients (each with a visits[] history)
def make_patient(pid):
    first, last = random.choice(first_names), random.choice(last_names)
    dob = rand_dob()
    group = random.choice(["A", "B", "AB", "O"]); rh = random.choice(["+", "-"])
    # 1-5 prior visits spread over the past ~3 years
    dates  = sorted(REF_DATE - timedelta(days=random.randint(30, 365*3)) for _ in range(random.randint(1, 5)))
    visits = [make_visit(d) for d in dates]
    return {
        "patient_id": pid,
        "last_name": last, "first_name": first,
        "date_of_birth": fmt(dob), "age": age_from(dob),
        "gender": random.choice(genders),
        "ssn": rand_ssn(),
        "insurance_number": rand_insurance(),
        "phone_number": rand_phone(),
        "blood_type": group + rh,
        "address": rand_address(),
        "email": make_email(first, last),
        "allergies": rand_allergies(),
        "medical_history": rand_history(),
        "current_medications": rand_meds(),
        "emergency_contact_name": random.choice(first_names) + " " + random.choice(last_names),
        "emergency_contact_phone": rand_phone(),
        "visits": visits,
    }

patients = [make_patient(f"P{i:04d}") for i in range(1, N_PATIENTS + 1)]
print("generated", len(patients), "patients")

In [ ]:
# Cell 5 — hold out the current visit for the first 100 patients; build CSV rows
validation   = patients[:N_VALIDATION]
VISIT_FIELDS = {"date_of_visit", "department", "doctor_name", "chief_complaint"}

csv_rows, held_out = [], {}
for p in validation:
    last_prior = max(parse(v["date_of_visit"]) for v in p["visits"])
    earliest   = max(last_prior + timedelta(days=1), REF_DATE - timedelta(days=60))
    dov        = earliest + timedelta(days=random.randint(0, max((REF_DATE - earliest).days, 0)))
    cur        = make_visit(dov)             # the NEW visit — stays OUT of the db
    held_out[p["patient_id"]] = cur

    row = {"patient_id": p["patient_id"]}
    for f in FORM_FIELDS:
        row[f] = cur[f] if f in VISIT_FIELDS else p.get(f, "")
    csv_rows.append(row)

print("built", len(csv_rows), "validation form rows (current visit held out of db)")

In [ ]:
# Cell 6 — write outputs
GEN.mkdir(parents=True, exist_ok=True)

with open(GEN / "patient_db.json", "w") as f:
    json.dump(patients, f, indent=2, default=str)

cols = ["patient_id"] + FORM_FIELDS
with open(GEN / "validation_forms.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(csv_rows)

print("wrote", GEN / "patient_db.json")
print("wrote", GEN / "validation_forms.csv")

In [ ]:
# Cell 7 — sanity checks
db = json.load(open(GEN / "patient_db.json"))
assert len(db) == N_PATIENTS, "patient count"
for pid, cur in held_out.items():
    pdb = next(p for p in db if p["patient_id"] == pid)
    assert all(not (v["date_of_visit"] == cur["date_of_visit"]
                    and v["chief_complaint"] == cur["chief_complaint"])
               for v in pdb["visits"]), f"held-out visit leaked into db for {pid}"

vmin = min(len(p["visits"]) for p in db); vmax = max(len(p["visits"]) for p in db)
print(f"OK  {len(db)} patients | visits/patient: {vmin}-{vmax} | validation held-out: {len(held_out)}")
print("CSV columns (form order):")
print(cols)
print()
print("Example db patient:")
import pprint; pprint.pprint(db[0])